In [5]:
import pandas as pd
import time
from pymongo import MongoClient
from collections import OrderedDict
import os

# --- CONFIGURACIÓN ---
NOMBRE_GRUPO = "G9-AgroTech"
NOMBRE_INTEGRANTE = "Sebastián Castillo"
NOMBRE_ARCHIVO = "precio-consumidor_semanal_202501-202552.xlsx" 

# --- PASO 1: LECTURA ---
try:
    if NOMBRE_ARCHIVO.endswith('.xlsx'):
        df = pd.read_excel(NOMBRE_ARCHIVO, skiprows=4)
    else:
        df = pd.read_csv(NOMBRE_ARCHIVO, skiprows=4)
    
    # Seleccionamos las columnas originales del Excel
    columnas_originales = ['Fecha inicio', 'Producto', 'Variedad', 'Calidad', 'Precio promedio']
    df_final = df[columnas_originales].copy()
    
    # Limpiamos filas vacías
    df_final = df_final.dropna(subset=['Producto'])
    print(f"✅ Archivo cargado correctamente: {len(df_final)} registros.")

except Exception as e:
    print(f"❌ Error al procesar el archivo: {e}")
    df_final = None

# --- PASO 2: CARGA A MONGODB CON RENOMBRADO ---
if df_final is not None:
    try:
        client = MongoClient('mongodb://bigdata_mongodb:27017/')
        db = client['previa_proyecto']
        coleccion = db['naranja']
        
        registros = df_final.to_dict(orient='records')
        exitos = 0
        
        print("--- Iniciando carga en colección 'naranja' ---")
        
        for fila in registros:
            try:
                # Usamos OrderedDict para controlar el orden de los campos
                doc = OrderedDict()
                doc["grupo"] = NOMBRE_GRUPO
                doc["Integrante"] = NOMBRE_INTEGRANTE
                
                # RENOMBRADO AQUÍ: De 'Fecha inicio' (Excel) a 'fecha' (Mongo)
                doc["Fecha"] = fila["Fecha inicio"]
                
                doc["Producto"] = fila["Producto"]
                doc["Variedad"] = fila["Variedad"]
                doc["Calidad"] = fila["Calidad"]
                
                # Limpieza de precio
                precio_raw = str(fila["Precio promedio"]).replace('.', '').replace(',', '.')
                doc["Precio promedio"] = float(precio_raw)
                
                # Columna de la extrema derecha
                doc["Fecha de captura"] = time.strftime("%Y-%m-%d %H:%M:%S")
                
                coleccion.insert_one(doc)
                exitos += 1
            except:
                continue
        
        print(f"✅ CARGA EXITOSA: {exitos} registros guardados con la columna 'fecha'.")
        client.close()
        
    except Exception as e:
        print(f"❌ Error de conexión: {e}")

❌ Error inesperado: "['Fecha'] not in index"


/opt/conda/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [6]:
# Diccionario para mapear meses en español a números
meses_map = {
    'ene.': '01', 'feb.': '02', 'mar.': '03', 'abr.': '04', 
    'may.': '05', 'jun.': '06', 'jul.': '07', 'ago.': '08', 
    'sep.': '09', 'oct.': '10', 'nov.': '11', 'dic.': '12'
}

# 1. Separamos el mes del año
df_split = df_urea['Mes'].str.split(' ', expand=True)

# 2. Reemplazamos el nombre del mes por su número
df_split[0] = df_split[0].map(meses_map)

# 3. Creamos la columna de fecha real
df_urea['Fecha'] = pd.to_datetime(df_split[1] + '-' + df_split[0] + '-01')

# 4. Ordenamos por fecha (por si el scraping los trajo desordenados)
df_urea = df_urea.sort_values('Fecha')

print(df_urea[['Fecha', 'Precio_CLP']].head())

       Fecha  Precio_CLP
0 2006-03-01    128560.1
1 2006-04-01    128038.3
2 2006-05-01    119470.6
3 2006-06-01    113916.6
4 2006-07-01    111097.5


In [4]:
import pandas as pd

# Asumiendo que df_potasio es el DataFrame que ya obtuvimos con el scrapping exitoso
def formatear_dataframe_big_data(df):
    # 1. Diccionario de mapeo para meses en español (formato IndexMundi)
    meses_map = {
        'ene.': '01', 'feb.': '02', 'mar.': '03', 'abr.': '04', 
        'may.': '05', 'jun.': '06', 'jul.': '07', 'ago.': '08', 
        'sep.': '09', 'oct.': '10', 'nov.': '11', 'dic.': '12'
    }

    # Creamos una copia para no alterar el original por accidente
    df_clean = df.copy()

    # 2. Procesamiento de Fecha
    # Separamos "mar. 2006" en ["mar.", "2006"]
    temp_split = df_clean['Mes'].str.split(' ', expand=True)
    
    # Mapeamos el nombre del mes a su número
    mes_num = temp_split[0].map(meses_map)
    año = temp_split[1]

    # Creamos la nueva columna Fecha (formato YYYY-MM-DD)
    df_clean['Fecha'] = pd.to_datetime(año + '-' + mes_num + '-01')

    # 3. Reorganizamos las columnas para que sea más legible
    # Dejamos la Fecha primero, luego el Precio y al final la Variación
    df_final = df_clean[['Fecha', 'Precio_CLP', 'Variacion']].sort_values('Fecha')

    return df_final

# --- Ejecución ---
df_listo = formatear_dataframe_big_data(df_potasio)

print("✅ DataFrame estructurado para análisis de series de tiempo:")
print(df_listo.head(15))

# Verificamos los tipos de datos (Dtype)
# print(df_listo.info())

✅ DataFrame estructurado para análisis de series de tiempo:
        Fecha  Precio_CLP  Variacion
0  2006-03-01    88569.12       0.00
1  2006-04-01    86652.16      -2.16
2  2006-05-01    87195.31       0.63
3  2006-06-01    90862.05       4.21
4  2006-07-01    93257.03       2.64
5  2006-08-01   103694.50      11.19
6  2006-09-01   103690.60       0.00
7  2006-10-01   102208.80      -1.43
8  2006-11-01   101493.00      -0.70
9  2006-12-01   101469.00      -0.02
10 2007-01-01   104139.30       2.63
11 2007-02-01   104386.30       0.24
12 2007-03-01   103658.90      -0.70
13 2007-04-01   102467.90      -1.15
14 2007-05-01   100475.70      -1.94
